In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences

MODEL_DIR = Path("artifacts")

# Load the trained model
loaded_model = tf.keras.models.load_model(
    MODEL_DIR / "clickbait_lstm.keras",
    compile=False
)

# Load the fitted tokeniser
with open(MODEL_DIR / "tokenizer.json", encoding="utf-8") as f:
    loaded_tokenizer = tokenizer_from_json(f.read())

# Load preprocessing configuration
with open(MODEL_DIR / "config.json", encoding="utf-8") as f:
    loaded_config = json.load(f)

print("Model and preprocessing artifacts loaded successfully.")

Model and preprocessing artifacts loaded successfully.


In [2]:
# Define example headlines
headlines = [
    "You Won't Believe What Happened Next",
    "Central Bank Keeps Interest Rates Unchanged"
]

# Tokenise and pad headlines
sequences = loaded_tokenizer.texts_to_sequences(headlines)

padded = pad_sequences(
    sequences,
    maxlen=loaded_config["max_length"],
    padding=loaded_config["padding"],
    truncating=loaded_config["truncating"]
)

# Generate predictions
probabilities = loaded_model.predict(
    padded, verbose=0
).ravel()

# Display results
pd.DataFrame({
    "headline": headlines,
    "clickbait_probability": probabilities.round(4),
    "predicted_class": np.where(
        probabilities >= loaded_config["threshold"],
        loaded_config["class_labels"]["1"],
        loaded_config["class_labels"]["0"]
    )
})

,headline,clickbait_probability,predicted_class
0,You Won't Believe What Happened Next,0.9995,Clickbait
1,Central Bank Keeps Interest Rates Unchanged,0.0040,Non-clickbait


In [3]:
from importlib.metadata import version

packages = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "tensorflow"
]

for package in packages:
    print(f"{package}=={version(package)}")

numpy==2.2.5
pandas==2.3.3
matplotlib==3.10.6
seaborn==0.13.2
scikit-learn==1.7.1
tensorflow==2.20.0
